In [ ]:
%load_ext autoreload
%autoreload 2

import pickle, json
import numpy as np
from go_bench.load_tools import load_protein_sequences

# Path to CAFA5 annotation data (not tracked in git).
# go_terms.json has been copied to ../../data/go_terms.json.
# prot_ids.json, rev_annot.pkl, and uniprot_sprot.fasta must be obtained
# from the CAFA5 dataset separately.
train_path = "../../data/cafa5"   # update if data is stored elsewhere

with open(f"{train_path}/go_terms.json", "r") as f:
    go_terms = json.load(f)
with open(f"{train_path}/prot_ids.json", "r") as f:
    prot_ids = json.load(f)
with open(f"{train_path}/rev_annot.pkl", "rb") as f:
    labels = pickle.load(f)

prot_sequences, seq_ids = load_protein_sequences(f"{train_path}/uniprot_sprot.fasta", set(prot_ids))
assert all(s1 == s2 for s1, s2 in zip(prot_ids, seq_ids))
labeled_id = (np.asarray(labels.sum(axis=1)) > 0).flatten()
gen = np.random.default_rng(seed=42)
ind = gen.permuted(np.arange(len(prot_ids))[labeled_id])
train_len = ind.shape[0] * 4 // 5
train_ind = ind[:train_len]
val_ind = ind[train_len:]
np.sort(train_ind); np.sort(val_ind)
train_ids = [prot_ids[i] for i in train_ind]; train_sequences = [prot_sequences[i] for i in train_ind]
val_ids = [prot_ids[i] for i in val_ind]; val_sequences = [prot_sequences[i] for i in val_ind]
train_labels = labels[train_ind, :]; val_labels = labels[val_ind, :]

In [2]:
from transformers import AutoTokenizer
esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D", do_lower_case=False)
from go_ml.data_utils import ProtFuncDataset, prot_func_collate
train_dataset = ProtFuncDataset(train_ids, train_sequences, train_labels, tokenizer=esm_tokenizer)
val_dataset = ProtFuncDataset(val_ids, val_sequences, val_labels, tokenizer=esm_tokenizer)
import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "wb") as f:
    pickle.dump(train_dataset, f)
with open(f"{data_path}/val_dataset.pkl", "wb") as f:
    pickle.dump(val_dataset, f)

In [3]:
from go_ml.data_utils import BertFuncDataset, prot_func_collate_bert
train_bert_dataset = BertFuncDataset.from_prot_func_dataset(train_dataset)


In [115]:
from torch.utils.data import Dataset, DataLoader
from go_ml.data_utils import ProtFuncDataset, prot_func_collate
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=prot_func_collate)
print(next(iter(val_loader)).keys())

dict_keys(['prot_id', 'seq_ind', 'mask', 'labels'])


In [116]:
#Time iterating through val_loader
import time
start = time.time()
for batch in val_loader:
    pass
end = time.time()
print(f"Time taken to iterate through val_loader: {end - start:.2f} seconds")

Time taken to iterate through val_loader: 3.25 seconds
